# Приоритизация обращений
Выполнил: Николай Шадрин

GitHub: https://github.com/NickShad/Avito_DS_test

## Что изменено по сравнению с baseline:
1. LightGBM лучше работает с табличными данными и дисбалансом.
2. Feature Engineering из events.csv агрегации активности пользователя за разные временные окна.
3. Временная валидация строгий split по дате назначения обращения.
4. Добавлен Optuna для автоматического поиска лучших гиперпараметров модели. 
    - По умолчанию есть сохранённая модель и Optuna отключен.
    - Можно изменить **OPTUNA_STUDY = True** для обновления гиперпараметров. После обучения новая модель сохраняется
5. Расширенные метрики: DailyAP, AP, AUC-ROC, LogLoss, Recall, Accuracy для полной диагностики. 
6. Вывод Submission.csv

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
import optuna
import joblib
from typing import Tuple, Dict, Any

# Sklearn
from sklearn.model_selection import TimeSeriesSplit, train_test_split
from sklearn.metrics import (
    average_precision_score, roc_auc_score, accuracy_score,
    recall_score, precision_score, f1_score, log_loss,
    precision_recall_curve, auc
)
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline

# LightGBM
import lightgbm as lgb

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

# Обучение optuna
OPTUNA_STUDY = False
N_TRIALS = 100
MODEL_DIR = Path(".") / "artifacts"
MODEL_DIR.mkdir(exist_ok=True)

MODEL_PATH = MODEL_DIR / "best_model.pkl"
PARAMS_PATH = MODEL_DIR / "best_params.pkl"
ENCODER_PATH = MODEL_DIR / "ordinal_encoder.pkl"

# Глобальные константы
RANDOM_STATE = 42
ROOT = Path(".")
DATA_DIR = ROOT / "data"
TARGET = "target"

np.random.seed(RANDOM_STATE)


c:\Users\kolya\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Загрузка и первичный анализ данных

Загружаем данные из файлов.

In [2]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
events = pd.read_csv(DATA_DIR / "events.csv")

print(f"Train shape: {train.shape}")
print(f"Test shape:  {test.shape}")
print(f"Events shape: {events.shape}\n")

print(f"Target mean: {train[TARGET].mean():.4f} - Дисбаланс классов")
print(f"Dates range: {train['assignment_date'].min()} - {train['assignment_date'].max()} - Промежуток времени")


Train shape: (13694, 119)
Test shape:  (4306, 118)
Events shape: (254705, 7)

Target mean: 0.2075 - Дисбаланс классов
Dates range: 2026-04-07 - 2026-04-22 - Промежуток времени


## 2. Инженерия признаков из events.csv

Используем данные из events.csv для обучения с учетом поведения пользователя.

Создаем признаки активности пользователя только до момента назначения обращения.

In [3]:
def daily_average_precision(y_true: np.ndarray, 
                           y_pred: np.ndarray, 
                           dates: pd.Series) -> float:
    """
    Считает Daily Average Precision.
    Используем numpy-массивы, чтобы избежать ошибки индексов в pandas.
    """
    y_true_arr = np.asarray(y_true)
    y_pred_arr = np.asarray(y_pred)
    dates_arr = np.asarray(dates)
    
    unique_dates = np.unique(dates_arr)
    if len(unique_dates) == 0:
        return 0.0
    
    daily_aps = []
    for date in unique_dates:
        mask = dates_arr == date
        y_true_day = y_true_arr[mask]
        y_pred_day = y_pred_arr[mask]
        
        # Пропускаем дни без позитивных примеров
        if y_true_day.sum() == 0:
            continue
        
        ap_day = average_precision_score(y_true_day, y_pred_day)
        daily_aps.append(ap_day)
    
    if len(daily_aps) == 0:
        return 0.0
    
    return float(np.mean(daily_aps))

def create_event_features(leads_df: pd.DataFrame, events_df: pd.DataFrame) -> pd.DataFrame:
    """
    Генерирует агрегированные признаки из событий для каждого обращения.
    Используем события, произошедшие строго до assignment_ts - Это предотвращает утечку данных из будущего.
    """
    leads = leads_df.copy()
    evts = events_df.copy()
    
    # Приводим даты к единому формату
    evts['event_ts'] = pd.to_datetime(evts['event_ts'])
    leads['assignment_ts'] = pd.to_datetime(leads['assignment_ts'])
    
    # Мерджим события с обращениями по user_id
    merged = evts.merge(
        leads[['lead_id', 'user_id', 'assignment_ts']],
        on='user_id', how='inner'
    )
    
    # Фильтруем события только до назначения
    merged = merged[merged['event_ts'] < merged['assignment_ts']]
    
    if len(merged) == 0:
        print(" Нет событий до назначения.")
        return leads
    
    # Вычисляем давность событий в днях
    merged['days_before'] = (merged['assignment_ts'] - merged['event_ts']).dt.days
    
    features_list = []
    
    # Агрегации за разные временные окна
    for window in [1, 3, 7, 14, 30]:
        w_data = merged[merged['days_before'] <= window]
        
        agg = w_data.groupby('user_id').agg(
            **{
                f'event_count_{window}d': ('event_ts', 'count'),
                f'unique_days_{window}d': ('event_ts', 'nunique'),
                f'unique_types_{window}d': ('event_type', 'nunique'),
            }
        ).reset_index()
        
        # Нормализуем на количество активных дней
        agg[f'event_rate_{window}d'] = (
            agg[f'event_count_{window}d'] / 
            agg[f'unique_days_{window}d'].clip(lower=1)
        )
        
        features_list.append(agg)
    
    # Recency: сколько дней прошло с последнего события
    recency = merged.groupby('user_id')['days_before'].min().reset_index()
    recency.columns = ['user_id', 'recency_days']
    
    # Diversity: энтропия типов событий 
    def calc_entropy(x):
        if len(x) == 0: return 0.0
        probs = x.value_counts(normalize=True)
        return -(probs * np.log2(probs + 1e-10)).sum()
    
    diversity = merged.groupby('user_id')['event_type'].apply(calc_entropy).reset_index()
    diversity.columns = ['user_id', 'event_entropy']
    
    # Собираем все признаки в один DataFrame
    result = leads[['user_id', 'lead_id']].copy()
    for feat in features_list:
        result = result.merge(feat, on='user_id', how='left')
    result = result.merge(recency, on='user_id', how='left')
    result = result.merge(diversity, on='user_id', how='left')
    
    # Заполняем NaN нулями (нет событий = нулевая активность)
    numeric_cols = result.select_dtypes(include=[np.number]).columns
    result[numeric_cols] = result[numeric_cols].fillna(0)
    
    n_new_features = len(result.columns) - 2  # минус user_id и lead_id
    print(f" Создано {n_new_features} новых признаков из событий")
    
    return result

# Применяем feature engineering к train и test
train_events = create_event_features(train, events)
test_events = create_event_features(test, events)


 Создано 22 новых признаков из событий
 Создано 22 новых признаков из событий


## 3. Объединение признаков и подготовка датасетов

In [4]:
# Исключаемые колонки
ID_COLUMNS = {'lead_id', 'user_id'}
TIME_COLUMNS = {'assignment_ts', 'assignment_date'}
NON_FEATURE_COLUMNS = ID_COLUMNS | TIME_COLUMNS | {TARGET, 'split'}

# Базовые признаки из train/test.csv
base_feature_columns = [
    col for col in train.columns
    if col not in NON_FEATURE_COLUMNS and col in test.columns
]

# Исключаем идентификаторы
event_feature_columns = [
    col for col in train_events.columns
    if col not in {'lead_id', 'user_id'} and col in test_events.columns
]

ALL_FEATURES = base_feature_columns + event_feature_columns

# Разделяем признаки по типу
# без обращения к train для event-признаков
numeric_columns = []
categorical_columns = []

for col in ALL_FEATURES:
    # Event-признаки всегда числовые
    if col in event_feature_columns:
        numeric_columns.append(col)
    # Базовые признаки проверяем по исходному train
    elif pd.api.types.is_numeric_dtype(train[col]):
        numeric_columns.append(col)
    else:
        categorical_columns.append(col)

print(f"\nВсего признаков: {len(ALL_FEATURES)}")
print(f"Числовых: {len(numeric_columns)}")
print(f"Категориальных: {len(categorical_columns)}")

# Объединяем базовые и событийные признаки
train_full = train.merge(train_events[['lead_id'] + event_feature_columns], on='lead_id', how='left')
test_full = test.merge(test_events[['lead_id'] + event_feature_columns], on='lead_id', how='left')

# Заполняем NaN в событийных признаках нулями
for col in event_feature_columns:
    train_full[col] = train_full[col].fillna(0)
    test_full[col] = test_full[col].fillna(0)

# Заполняем пропуски в числовых признаках медианой на train_part
for col in numeric_columns:
    median_val = train_full[col].median()
    train_full[col] = train_full[col].fillna(median_val)
    test_full[col] = test_full[col].fillna(median_val)

# Кодирование категориальных признаков
ordinal_encoder = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=-1,
    dtype=np.int32
)

# Обучаем на train_full
train_full[categorical_columns] = ordinal_encoder.fit_transform(train_full[categorical_columns])
test_full[categorical_columns] = ordinal_encoder.transform(test_full[categorical_columns])


Всего признаков: 136
Числовых: 129
Категориальных: 7


## 4. Временная валидация

Делим данные по времени: ранние даты для train, поздние для validation.

In [5]:
def make_validation_split(df: pd.DataFrame, val_ratio: float = 0.2) -> tuple:
    dates = pd.to_datetime(df['assignment_date']).dt.date
    ordered_dates = sorted(dates.unique())
    cutoff_idx = int(len(ordered_dates) * (1 - val_ratio))
    cutoff_date = ordered_dates[cutoff_idx]
    
    train_part = df[dates < cutoff_date].copy()
    val_part = df[dates >= cutoff_date].copy()
    
    print(f"\n   Временной split:")
    print(f"   Cutoff date: {cutoff_date}")
    print(f"   Train: {train_part.shape[0]} строк ({train_part[TARGET].mean():.4f} target mean)")
    print(f"   Val:   {val_part.shape[0]} строк ({val_part[TARGET].mean():.4f} target mean)")
    
    return train_part, val_part

train_part, val_part = make_validation_split(train_full)


   Временной split:
   Cutoff date: 2026-04-19
   Train: 10272 строк (0.2081 target mean)
   Val:   3422 строк (0.2054 target mean)


## 5. Preprocessor для LightGBM

Указываем точно типы категориальных признаков для работы LightGBM.

In [6]:
# Указываем категориальные колонки для LightGBM 
cat_features_lgb = categorical_columns

# Для числовых признаков заполняем пропуски медианой
for col in numeric_columns:
    median_val = train_part[col].median()
    train_part[col] = train_part[col].fillna(median_val)
    val_part[col] = val_part[col].fillna(median_val)
    test_full[col] = test_full[col].fillna(median_val)

# Для категориальных заполняем модой
for col in categorical_columns:
    mode_val = train_part[col].mode()[0] if len(train_part[col].mode()) > 0 else 'unknown'
    train_part[col] = train_part[col].fillna(mode_val)
    val_part[col] = val_part[col].fillna(mode_val)
    test_full[col] = test_full[col].fillna(mode_val)

X_train = train_part[ALL_FEATURES]
y_train = train_part[TARGET]
X_val = val_part[ALL_FEATURES]
y_val = val_part[TARGET]
X_test = test_full[ALL_FEATURES]

# Вес для дисбаланса классов
scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
print(f"\nScale pos weight: {scale_pos_weight:.2f}")



Scale pos weight: 3.80


## 6. Поиск оптимальных гиперпараметров через Optuna

In [7]:
if OPTUNA_STUDY:
    print("Обучение через Optuna")
        
    def objective(trial: optuna.Trial) -> float:
        """Функция цели для Optuna. Возвращает Daily Average Precision на валидации."""
        params = {
            'objective': 'binary',
            'metric': 'average_precision',
            'boosting_type': trial.suggest_categorical('boosting_type', ['gbdt', 'dart']),
            'num_leaves': trial.suggest_int('num_leaves', 20, 150),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
            'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
            'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 200),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            'scale_pos_weight': scale_pos_weight,
            'verbose': -1,
            'random_state': RANDOM_STATE,
            'n_jobs': -1,
        }
        
        model = lgb.LGBMClassifier(**params, n_estimators=2000)
        
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            callbacks=[
                lgb.early_stopping(stopping_rounds=80, verbose=False),
                lgb.log_evaluation(period=0)
            ],
            categorical_feature=categorical_columns
        )
        
        preds = model.predict_proba(X_val)[:, 1]
        val_dates = val_part['assignment_date']
        daily_ap = daily_average_precision(y_val, preds, val_dates)
        
        return daily_ap

    print("\nЗапуск Optuna")
    study = optuna.create_study(direction='maximize', study_name='lead_prioritization')
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

    print(f"\nBest Daily AP: {study.best_value:.5f}")
    print(f"Best params: {study.best_params}")

    best_params = study.best_params.copy()
    best_params.update({
        'objective': 'binary',
        'metric': 'average_precision',
        'scale_pos_weight': scale_pos_weight,
        'verbose': -1,
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
    })

else:
    print("Загрузка сохранённой модели")
    
    assert MODEL_PATH.exists(), f"Модель не найдена: {MODEL_PATH}. Сначала обучите с OPTUNA_STUDY=True"
    assert PARAMS_PATH.exists(), f"Параметры не найдены: {PARAMS_PATH}"
    assert ENCODER_PATH.exists(), f"Энкодер не найден: {ENCODER_PATH}"
    
    final_model = joblib.load(MODEL_PATH)
    best_params = joblib.load(PARAMS_PATH)
    ordinal_encoder = joblib.load(ENCODER_PATH)

    print(f"Загружена модель: {MODEL_PATH}")
    print(f"Параметры: {best_params}")
    print(f"Энкодер: {ENCODER_PATH}")

Загрузка сохранённой модели
Загружена модель: artifacts\best_model.pkl
Параметры: {'boosting_type': 'gbdt', 'num_leaves': 41, 'learning_rate': 0.014356784696045736, 'feature_fraction': 0.5220645226972641, 'bagging_fraction': 0.8428034790092835, 'bagging_freq': 1, 'min_child_samples': 90, 'reg_alpha': 0.0035308999642343544, 'reg_lambda': 0.011365140542966773, 'objective': 'binary', 'metric': 'average_precision', 'scale_pos_weight': 3.804490177736202, 'verbose': -1, 'random_state': 42, 'n_jobs': -1}
Энкодер: artifacts\ordinal_encoder.pkl


## 7. Обучение финальной модели и полная оценка метрик

In [8]:
"""Вывод метрик для оценки обучения модели"""

def evaluate_model(y_true: np.ndarray, 
                  y_pred_proba: np.ndarray, 
                  dates: pd.Series, 
                  name: str = "") -> Dict[str, Any]:
    # Сохраняем результат
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_pred_proba)
    
    # Оптимальный порог по F1
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
    best_idx = np.argmax(f1_scores)
    optimal_threshold = thresholds[min(best_idx, len(thresholds)-1)]
    
    y_pred = (y_pred_proba >= optimal_threshold).astype(int)
    
    metrics = {
        'Average Precision (AP)': average_precision_score(y_true, y_pred_proba),
        'Daily Average Precision': daily_average_precision(y_true, y_pred_proba, dates),
        'AUC-ROC': roc_auc_score(y_true, y_pred_proba),
        'PR-AUC': auc(recalls, precisions),
        'Accuracy': accuracy_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'F1-Score': float(f1_scores[best_idx]),
        'Log Loss': log_loss(y_true, y_pred_proba),
        'Optimal Threshold (by F1)': float(optimal_threshold),
    }
    
    n_pos = int(np.sum(y_true))
    metrics['Class balance'] = f"{n_pos}/{len(y_true)} ({n_pos/len(y_true)*100:.1f}% positive)"
    
    print(f"\n{'='*60}")
    print(f" МЕТРИКИ{name}")
    print(f"{'='*60}")
    for k, v in metrics.items():
        print(f"   {k:35s}: {v}")
    print(f"{'='*60}")
    
    return metrics


# Обучаем модель с лучшими параметрами на train_part
final_model = lgb.LGBMClassifier(**best_params, n_estimators=2000)
final_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=80),
        lgb.log_evaluation(period=100)
    ],
    categorical_feature=cat_features_lgb
)

if OPTUNA_STUDY:
    print("\nСохранение модели")
    joblib.dump(final_model, MODEL_PATH)
    joblib.dump(best_params, PARAMS_PATH)
    joblib.dump(ordinal_encoder, ENCODER_PATH)
    print(f"   Модель: {MODEL_PATH}")
    print(f"   Параметры: {PARAMS_PATH}")
    print(f"   Энкодер: {ENCODER_PATH}")

# Оценка на валидации
val_preds = final_model.predict_proba(X_val)[:, 1]

# Выводим метрики
val_metrics = evaluate_model(
    y_true=y_val.values, 
    y_pred_proba=val_preds, 
    dates=val_part['assignment_date'],
    name=" (Validation)"
)

# Считаем Daily AP отдельно
val_daily_ap = daily_average_precision(
    y_true=y_val.values, 
    y_pred=val_preds, 
    dates=val_part['assignment_date']
)
print(f"\nDaily AP: {val_metrics['Daily Average Precision']:.5f}")

# Сравнение с глобальным AP для диагностики
val_global_ap = val_metrics['Average Precision (AP)']
print(f"Global AP: {val_global_ap:.5f}")
print(f"Разница (Global - Daily): {val_global_ap - val_daily_ap:.5f}")

"""Разница показывает, насколько хорошо модель ранжирует данные в каждый день
в сравнении с ранжированием для всего времени и стабильность самой модели. 

так как значение на Daily AP  не меньше, чем на Global, то это значит,
что модель не переобучилась для дней с большим количеством данных."""

# Важность признаков
importance = pd.DataFrame({
    'feature': ALL_FEATURES,
    'gain': final_model.feature_importances_
}).sort_values('gain', ascending=False)

print("\n Топ-20 важных признаков:")
print(importance.head(20).to_string(index=False))


Training until validation scores don't improve for 80 rounds
[100]	valid_0's average_precision: 0.514119
[200]	valid_0's average_precision: 0.54176
[300]	valid_0's average_precision: 0.557655
[400]	valid_0's average_precision: 0.567925
[500]	valid_0's average_precision: 0.575553
[600]	valid_0's average_precision: 0.580384
[700]	valid_0's average_precision: 0.584299
[800]	valid_0's average_precision: 0.585033
[900]	valid_0's average_precision: 0.585491
[1000]	valid_0's average_precision: 0.587152
[1100]	valid_0's average_precision: 0.588421
[1200]	valid_0's average_precision: 0.588142
Early stopping, best iteration is:
[1126]	valid_0's average_precision: 0.58889

 МЕТРИКИ (Validation)
   Average Precision (AP)             : 0.5888899728875064
   Daily Average Precision            : 0.5939754845716667
   AUC-ROC                            : 0.8061379356166527
   PR-AUC                             : 0.5884338309231594
   Accuracy                           : 0.7869666861484512
   Recall   

## 8. Генерация submission.csv

In [9]:
print("\nОбучение на train для финального предсказания")

# Переобучаем на полном train с лучшим количеством итераций
full_model = lgb.LGBMClassifier(**best_params, n_estimators=final_model.best_iteration_)
full_model.fit(
    train_full[ALL_FEATURES], train_full[TARGET],
    categorical_feature=cat_features_lgb
)

test_scores = full_model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    'lead_id': test['lead_id'].astype(str),
    'score': test_scores
})

# Проверки формата
assert list(submission.columns) == ['lead_id', 'score']
assert len(submission) == len(test)
assert submission['lead_id'].is_unique
assert submission['score'].between(0, 1).all()

submission.to_csv(ROOT / "submission.csv", index=False)

print(f"\n{'='*60}")
print("submission.csv создан.")
print(f"   Строк: {len(submission)}")
print(f"   Score mean: {submission['score'].mean():.4f}")
print(f"   Score std:  {submission['score'].std():.4f}")
print(f"{'='*60}")
print(submission.head(10).to_string(index=False))


Обучение на train для финального предсказания

submission.csv создан.
   Строк: 4306
   Score mean: 0.3229
   Score std:  0.2395
              lead_id    score
lead_97e409eb8f8c8246 0.795418
lead_55310edb4489f9e9 0.526542
lead_e7f653a2c6a7eee8 0.949001
lead_22f8e1cfc487ac20 0.103435
lead_48b638b839abfac3 0.195707
lead_bf0b08db8abc5aa5 0.443732
lead_a576da4143978645 0.430482
lead_b8088850f04f3c69 0.079491
lead_5c4a70a402d17d14 0.024246
lead_364d2ff08df16ea6 0.441892
